# 03 — Contamination Placebo

Gated by RUN_PLACEBO=True env var AND smoke-test gate.
Runs three conditions: original / anonymized / shuffled. Reports IC-drop CI.

In [ ]:
import sys, os
from pathlib import Path
for _cand in ['.', '..']:
    if (Path(_cand)/'src').is_dir() and (Path(_cand)/'legacy').is_dir():
        os.chdir(_cand); break
sys.path.insert(0, 'src'); sys.path.insert(0, 'legacy/src')

RUN_PLACEBO = os.environ.get('RUN_PLACEBO', 'false').lower() == 'true'

if not RUN_PLACEBO:
    print('RUN_PLACEBO is not set — skipping notebook 03.')
    print('To run: set environment variable RUN_PLACEBO=true and re-execute.')
    import sys; sys.exit(0)

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.placebo import run_placebo_analysis
from src.stats_rigor import bootstrap_ic_ci
from src.grid import append_cells
from src.report_io import save_fig, save_table, setup_style
from src.config import PANELS_2016

setup_style()
N_BOOT    = 1000
N_EVENTS  = 300  # events per condition
SEED      = 42
print(f'Placebo analysis: n_events={N_EVENTS}, n_boot={N_BOOT}, seed={SEED}')

## 1. Load panel (use AAPL 2016-2020 for reproducibility)

In [ ]:
aapl_path = PANELS_2016.get('AAPL')
if not aapl_path or not Path(aapl_path).exists():
    raise RuntimeError(f'AAPL 2016-2020 panel not found at {aapl_path}')

panel = pd.read_csv(aapl_path)
panel['stock'] = 'AAPL'
print(f'Loaded AAPL 2016-2020 panel: {len(panel):,} rows')

## 2. Run three-condition placebo

Calls `score_llm()` from legacy/src/sentiment.py for anonymized and shuffled conditions.
This makes live Anthropic API calls — ensure ANTHROPIC_API_KEY is set.

In [ ]:
try:
    results = run_placebo_analysis(
        panel, ticker='AAPL', horizon=1,
        n_events=N_EVENTS, n_boot=N_BOOT, seed=SEED
    )
    print('Placebo analysis complete.')
    print(results)
except Exception as e:
    print(f'ERROR in placebo run: {e}')
    print('Ensure ANTHROPIC_API_KEY is set and the API is reachable.')
    results = None

## 3. IC-drop CI and figure

In [ ]:
if results is not None:
    # Expected keys from run_placebo_analysis:
    # ic_original, ic_anon, ic_shuffled,
    # ci_drop_anon, ci_drop_shuffled, p_drop_anon, p_drop_shuffled
    conditions   = ['original', 'anonymized', 'shuffled']
    ic_vals      = [results.get('ic_original', np.nan),
                    results.get('ic_anon',     np.nan),
                    results.get('ic_shuffled', np.nan)]
    ci_los = [results.get('ci_lo_original', np.nan),
              results.get('ci_lo_anon',     np.nan),
              results.get('ci_lo_shuffled', np.nan)]
    ci_his = [results.get('ci_hi_original', np.nan),
              results.get('ci_hi_anon',     np.nan),
              results.get('ci_hi_shuffled', np.nan)]

    fig, ax = plt.subplots(figsize=(6, 4))
    x = np.arange(len(conditions))
    ics = np.array(ic_vals)
    los = ics - np.array(ci_los)
    his = np.array(ci_his) - ics
    ax.bar(x, ics, color=['#2E86AB', '#5E8FAA', '#A23B72'],
           yerr=[los, his], error_kw={'linewidth': 0.8, 'capsize': 4})
    ax.set_xticks(x)
    ax.set_xticklabels(conditions)
    ax.axhline(0, color='black', lw=0.7, ls='--', alpha=0.5)
    ax.set_ylabel('Spearman IC (1-min)')
    ax.set_title('Contamination placebo: IC by headline condition')
    ax.grid(axis='y', alpha=0.2)
    save_fig(fig, '03_placebo_ic')
    plt.close()

    drop_tbl = pd.DataFrame({
        'condition':   ['original', 'anonymized', 'shuffled'],
        'ic':          ic_vals,
        'ci_lo':       ci_los,
        'ci_hi':       ci_his,
        'ic_drop_vs_original': [0,
                                 results.get('ic_original',0) - results.get('ic_anon',0),
                                 results.get('ic_original',0) - results.get('ic_shuffled',0)],
    })
    print(drop_tbl.to_string(index=False, float_format='{:.4f}'.format))
    save_table(
        drop_tbl,
        '03_placebo_ic_drop',
        caption='Contamination placebo: Spearman IC under three headline conditions (AAPL 2016-2020, 1-min horizon). Anonymized = entity names replaced; shuffled = wrong company + wrong date.',
        label='tab:placebo_ic',
    )

    # Append to grid
    grid_rows = []
    for cond, ic, lo, hi, p_key in [
        ('anon',     results.get('ic_anon'),     results.get('ci_lo_anon'),     results.get('ci_hi_anon'),     'p_drop_anon'),
        ('shuffled', results.get('ic_shuffled'), results.get('ci_lo_shuffled'), results.get('ci_hi_shuffled'), 'p_drop_shuffled'),
    ]:
        grid_rows.append({
            'notebook': '03', 'cell_id': f'placebo_{cond}',
            'stock': 'AAPL', 'scorer': 'llm_placebo', 'horizon': 1,
            'n': N_EVENTS, 'ic': ic, 'ci_lo': lo, 'ci_hi': hi,
            'p': results.get(p_key, np.nan),
        })
    append_cells(grid_rows)
    print(f'Appended {len(grid_rows)} cells to secondary grid.')
else:
    print('Skipping figure and grid append (placebo failed).')